Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Exercise: Getting Comfortable with Our Open Lakehouse

This first exercise is set up to make sure that your container environment correctly started up and that
Apache Iceberg tables are accessible from multiple engines.

## The Tools

- **Apache Iceberg**: Open table format that describes how to do SQL on files
- **RustFS**: S3-compatible object storage
- **Apache Polaris**: Iceberg REST Compatible Catalog
- **Spark**: Distributed Analytics Engine
- **Trino**: Distributed Analytics Engine

⚠️ **IMPORTANT**: This environment uses **Spark Connect** — a single Spark server runs in the background and all notebooks connect to it as lightweight clients (no per-notebook JVM). If you see `ConnectionRefusedError: [Errno 111] Connection refused` when initializing the Spark Session, the Spark Connect server may not be running. Check the Docker container logs with `docker logs jupyter-spark`. If the server failed to start, try restarting the container with `docker compose restart jupyter`.

## Explore the Environment


### Jupyter Notebooks

**You're here!**

Jupyter provides an interactive environment for running Spark and Python code. All the lesson exercises are stored in the `/notebooks` directory.

**What to explore:**
- The file browser on the left shows all available notebooks
- Each notebook corresponds to an exercise in the course
- The videos folder contains all of the examples shown during course videos
- You can create new notebooks to experiment with your own queries or code
- The integrated terminal allows you to run shell commands if needed

### Polaris Catalog

**URL:** Polaris UI is not yet released

Polaris is an Iceberg REST compatible catalog that manages access and governance for Iceberg tables. It associates table identifiers with specific Iceberg metadata. Unlike traditional databases where the catalog is built into the database server, Iceberg separates the catalog into its own service so that multiple engines (Spark, Trino, Flink) can share the same table definitions. The REST API is how those engines communicate with the catalog.

**What to explore:**
- List the available namespaces and tables in the Polaris catalog from Spark (we'll do this shortly)
- Inspect table metadata through the REST API or Spark SQL metadata tables

### Spark Configuration

**Configuration File:** `/home/jovyan/.sparkconf/spark-defaults.conf`

The Spark Session configuration is automatically generated by Docker Compose and is placed in the above file to wire together all of the components
of this Lakehouse as well as download necessary dependencies for Spark to work with Iceberg. The code below opens that file, reads it into the config variable, and then prints that variable for you to see.

**Key Configuration:**
- **Required Packages** Iceberg Spark Runtime and AWS Bundle
- **Catalog Configuration** - Points Spark to the Polaris REST catalog
- **S3 Credentials** - Authenticates with RustFS storage

In [1]:
with open('/home/jovyan/.sparkconf/spark-defaults.conf', 'r') as f:
    config = f.read()
    print(config)

# Copyright 2026 Snowflake Inc.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Iceberg Spark Dependencies
spark.jars.packages=org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,org.apache.iceberg:iceberg-aws-bundle:1.10.0,software.amazon.awssdk:bundle:2.20.18,software.amazon.awssdk:url-connection-client:2.20.18,org.apache.hadoop:hadoop-aws:3.4.1

# Iceberg Spark SQL/Catalyst Extensions
spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSpark

## Initialize Spark Session

Now let's start coding! We'll create a Spark session that connects to our pre-configured Iceberg environment. As you can see in the printed result, we are using the Polaris catalog.

In [2]:
from pyspark.sql import SparkSession

# All configuration is loaded from spark-defaults.conf
spark = SparkSession.builder \
    .appName("OpenLakehouse") \
    .getOrCreate()

print(f" Spark {spark.version} initialized!")
print(f" Default Catalog: {spark.conf.get('spark.sql.defaultCatalog', 'spark_catalog')}")

 Spark 4.0.1 initialized!
 Default Catalog: polaris


/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/base.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/commands.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/common.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/p

## Create Your First Iceberg Table

Let's create a simple table to demonstrate the lakehouse in action. We'll create a namespace (like a database schema) and a table within it.

In [3]:
print(spark.conf.get("spark.sql.catalog.polaris.uri"))
print(spark.conf.get("spark.sql.catalog.polaris.credential"))
print(spark.conf.get("spark.sql.catalog.polaris.warehouse"))

http://polaris:8181/api/catalog
root:s3cr3t
quickstart_catalog


In [4]:


# Create namespace (like a schema in traditional databases)
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.demo")
print(" Namespace 'demo' created!")

# Drop and recreate so re-running the notebook starts fresh
spark.sql("DROP TABLE IF EXISTS polaris.demo.employees")

# Create an Iceberg table
# USING iceberg tells Spark to create an Iceberg table rather than a plain
# Spark-managed table. Without it, Spark creates a non-Iceberg table that
# doesn't support snapshots, time travel, or schema evolution.
#
# Format versions: V1 = original spec, V2 = row-level deletes,
# V3 = delete vectors and new types like Variant and Geo.
# V1/V2 are still in wide use. V3 is recommended for new tables.
# Upgrading is a one-way metadata change with no data rewrite.
spark.sql("""
    CREATE TABLE polaris.demo.employees (
        id INT,
        name STRING,
        department STRING
    ) USING iceberg
    TBLPROPERTIES (
        'format-version' = '3'
    )
""")
print(" Table 'employees' created!")

 Namespace 'demo' created!
 Table 'employees' created!


## Insert Sample Data

Now with our table created and our initial schema set, let's insert a series of data values into this table.

In [5]:
spark.sql("""
    INSERT INTO polaris.demo.employees VALUES
    (1, 'Alice', 'Engineering'),
    (2, 'Bob', 'Sales'),
    (3, 'Charlie', 'Marketing'),
    (4, 'Diana', 'Engineering')
""")
print(" Sample data inserted!")

 Sample data inserted!


## Query with Spark

Now let's confirm that the data is in the table and read the data back using Spark SQL.

In [6]:
spark.sql("""
    SELECT * FROM polaris.demo.employees
""").show()

+---+-------+-----------+
| id|   name| department|
+---+-------+-----------+
|  1|  Alice|Engineering|
|  2|    Bob|      Sales|
|  3|Charlie|  Marketing|
|  4|  Diana|Engineering|
+---+-------+-----------+



In [7]:
spark.sql("""
    SELECT department, COUNT(*) as employee_count
    FROM polaris.demo.employees
    GROUP BY department
    ORDER BY employee_count DESC
""").show()

+-----------+--------------+
| department|employee_count|
+-----------+--------------+
|Engineering|             2|
|      Sales|             1|
|  Marketing|             1|
+-----------+--------------+



### Explore the Spark UI

**URL:** http://localhost:4040

The Spark UI provides detailed information about your Spark jobs:

**What you can see:**
- **Jobs tab**: Overview of all Spark jobs executed
- **Stages tab**: Breakdown of each job into stages and tasks
- **SQL tab**: Query plans and execution details for Spark SQL operations
- **Environment tab**: Configuration settings and runtime properties

This is useful for understanding query performance and debugging issues.

## Query with Trino

One of the key benefits of Apache Iceberg is **engine interoperability**. Traditional systems couple the storage layout to a specific engine. Iceberg defines the table structure in open metadata files that any engine can read, so the data is truly engine-independent.

Let's connect to Trino and query the exact same data. Trino is a distributed SQL query engine designed for analytics. It's completely separate from Spark, but both can read/write the same Iceberg tables. Trino is accessible via JDBC which we are using below.

In [8]:
from trino.dbapi import connect
import pandas as pd
import time

# Connect to Trino (with retry for container startup timing)
max_retries = 3
for attempt in range(max_retries):
    try:
        conn = connect(
            host='trino',
            port=8080,
            user='trino',
            catalog='polaris',
            schema='demo'
        )

        cursor = conn.cursor()

        # Query the same table we created with Spark!
        cursor.execute("SELECT * FROM employees")
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]

        # Display results
        df = pd.DataFrame(rows, columns=columns)
        print(" Querying via Trino:\n")
        print(df.to_string(index=False))

        cursor.close()
        conn.close()
        break
    except Exception as e:
        if attempt < max_retries - 1:
            print(f"Trino connection attempt {attempt + 1} failed, retrying in 5s...")
            time.sleep(5)
        else:
            raise

 Querying via Trino:

 id    name  department
  2     Bob       Sales
  3 Charlie   Marketing
  4   Diana Engineering
  1   Alice Engineering


### Explore the Trino UI

**URL:** http://localhost:8080  
**Credentials:** admin

The Trino UI shows query execution and cluster status:

**What you can see:**
- **Query details**: Click any query to see execution plan, data scanned, and performance metrics. Make sure to select *finished* in the filter to see the above query
- **Cluster overview**: Worker nodes and resource utilization
- **Live query monitoring**: Real-time progress of running queries

The UI is great for monitoring query performance and understanding how Trino processes your SQL.

## Explore RustFS

Now that we've created tables and inserted data, let's explore RustFS to see the actual files that were created!

**URL:** http://localhost:9001/browser/warehouse/demo/employees  
**Credentials:** admin / password

RustFS is an S3-compatible object storage where all data and metadata files are stored.

**What to explore:**
- Click into the `data/` folder to see the actual **Parquet** data files. Parquet is a columnar binary file format widely used in big data. Think of it as a compressed, column-oriented alternative to CSV. Iceberg uses Parquet as its default storage format because columnar layout enables efficient reads of individual columns without scanning entire rows.
- Click into the `metadata/` folder to see Iceberg Metadata Json, Manifest and Manifest List files

Iceberg's metadata forms a tree with several layers: **snapshot → manifest list → manifests → data files**. Why so many layers instead of having a snapshot point directly at data files? Each layer enables a different level of skipping during query planning:

- **Manifests** list individual data files along with their partition values and column-level statistics (min/max values). This lets engines skip irrelevant files without opening them.
- **Manifest lists** group manifests together with partition summaries, so engines can skip entire groups of files at once, which is critical when a table has thousands of files.
- **Snapshots** point to a manifest list, representing the complete state of the table at a point in time.

You'll see how this structure powers query optimization in the next exercise (E1.2).

## Congrats!

This concludes your first exercise. Now, you've learned to set up your Open Lakehouse by configuring a catalog and storage object and executing queries on them with engines like Trino and Apache Spark.